# UTS Data Acquisition & Management — Kelas E
**Nama:** (isi) | **NIM:** (isi) | **Dosen:** Dr. I Gede Totok Suryawan, S.Kom., M.T.

**Topik (sektor pariwisata):** Akomodasi Non-Bintang di Provinsi Bali, 2024

| Cara akuisisi | Sumber | Tujuan |
|---|---|---|
| (a) Berkas dataset terbuka | BPS Provinsi Bali (CSV) | Jumlah akomodasi non-bintang per kab/kota |
| (b) API | Open-Meteo Historical Weather API | Cuaca harian tiap kab/kota (konteks wisata) |
| (c) Web scraping | Wikipedia (halaman statis) | Data wilayah/penduduk kab/kota |

Semua hasil mentah disimpan di `data/raw` tanpa diubah. Setiap sumber dicatat provenance-nya (URL, tanggal akses, lisensi, MD5).

## 0. Setup
Persiapan library, folder, identitas request (User-Agent berisi email kampus), dan helper. **Email tidak ditulis di notebook**: simpan sebagai Colab Secret bernama `CAMPUS_EMAIL` (ikon kunci di sidebar kiri Colab), atau di berkas `.env`.

In [ ]:
# Instal library tambahan (python-dotenv untuk .env, lxml/bs4 untuk parsing HTML)
!pip -q install python-dotenv lxml beautifulsoup4

In [ ]:
# ===== 0. SETUP =====
import os, io, re, json, time, hashlib, shutil
from datetime import datetime
from pathlib import Path
from urllib import robotparser
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Struktur folder sesuai petunjuk: data/raw TIDAK boleh diubah isinya
RAW, PROC, PROV = Path("data/raw"), Path("data/processed"), Path("data/provenance")
for p in (RAW, PROC, PROV):
    p.mkdir(parents=True, exist_ok=True)

# Ambil nilai rahasia dari Colab Secrets; jika bukan di Colab, baca dari berkas .env
def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        from dotenv import load_dotenv
        load_dotenv(dotenv_path=".env")
        return os.getenv(name)

EMAIL = get_secret("CAMPUS_EMAIL")
assert EMAIL, "Isi Colab Secret CAMPUS_EMAIL (atau .env) dengan email kampus Anda dulu."

# Identitas request: User-Agent berisi email kampus (wajib), timeout, dan jeda antar-request
USER_AGENT = f"UTS-DAM-INSTIKI/1.0 (mahasiswa Informatika; kontak: {EMAIL})"
HEADERS = {"User-Agent": USER_AGENT}
TIMEOUT = 20   # detik, batas tunggu tiap request
DELAY = 2      # detik, jeda antar-request
print("Setup OK. Folder:", RAW, PROC, PROV)

In [ ]:
# ===== HELPER: MD5, provenance, dan request dengan retry =====
provenance = []   # daftar catatan provenance seluruh sumber

def md5_file(path):
    # Hitung hash MD5 berkas (dibaca per blok agar hemat memori)
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(8192), b""):
            h.update(blok)
    return h.hexdigest()

def catat_provenance(nama, metode, url, lisensi, path, catatan=""):
    # Simpan satu baris provenance: URL, tanggal akses, lisensi, hash MD5, ukuran
    provenance.append({
        "sumber": nama, "metode": metode, "url": url,
        "tanggal_akses": datetime.now().astimezone().isoformat(timespec="seconds"),
        "lisensi": lisensi, "berkas": str(path),
        "md5": md5_file(path), "ukuran_byte": Path(path).stat().st_size,
        "catatan": catatan})

def get_with_retry(url, params=None, max_retry=4):
    # GET dengan timeout + User-Agent; retry untuk 429/5xx dan error jaringan (backoff eksponensial)
    for percobaan in range(1, max_retry + 1):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=TIMEOUT)
            print(f"  percobaan {percobaan}: HTTP {r.status_code}")
            if r.status_code == 200:
                return r
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** percobaan)   # tunggu lalu coba lagi
                continue
            r.raise_for_status()             # 4xx lain: gagal, tidak di-retry
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"  percobaan {percobaan}: error jaringan ({type(e).__name__})")
            time.sleep(2 ** percobaan)
    raise RuntimeError(f"Gagal mengambil {url} setelah {max_retry} percobaan")

## Soal 1 — Membaca berkas, identifikasi dataset, estimasi storage
**Langkah:** (1) unggah CSV mentah ke `data/raw` (2) catat provenance (3) baca dengan parameter tepat (4) identifikasi (5) hitung storage.

In [ ]:
# 1a. Unggah berkas CSV hasil unduhan BPS ke data/raw (tanpa diubah isinya)
NAMA_FILE = "Jumlah_Hotel_Non_Bintang_dan_Akomodasi_Lainnya_Menurut_Kelompok_Kamar_dan_Kabupaten_Kota_di_Provinsi_Bali__2024.csv"
RAW_FILE = RAW / NAMA_FILE
if not RAW_FILE.exists():
    from google.colab import files
    for nama in files.upload():          # dialog unggah muncul; pilih CSV Anda
        shutil.move(nama, RAW / nama)    # pindahkan apa adanya ke data/raw
print("Berkas mentah:", RAW_FILE.exists(), "| MD5:", md5_file(RAW_FILE))

In [ ]:
# 1b. Catat provenance berkas. ISI url & lisensi sesuai halaman sumber saat Anda mengunduh!
URL_BPS = "ISI_URL_HALAMAN_TABEL_BPS_ATAU_SATU_DATA_BALI"   # <-- ganti dengan URL halaman unduhan
LISENSI_BPS = "ISI_LISENSI_DARI_HALAMAN_SUMBER"             # <-- ganti, mis. ketentuan penggunaan data portal tsb
catat_provenance("BPS Provinsi Bali - Akomodasi non-bintang 2024", "unduh berkas", URL_BPS,
                 LISENSI_BPS, RAW_FILE, "CSV UTF-8 BOM, header 4 baris, '-' = tidak ada data")

In [ ]:
# 1c. Baca CSV dengan parameter yang tepat:
#  encoding="utf-8-sig" -> membuang BOM di awal berkas
#  skiprows=4, header=None, names=... -> 4 baris awal adalah judul/tahun, bukan data
#  na_values=["-"] -> tanda "-" berarti data kosong
#  dtype -> kolom angka Int64 (mendukung NaN), nama wilayah string
kolom = ["kabupaten_kota", "kamar_lt_10", "kamar_10_24", "kamar_25_40", "kamar_gt_41", "jumlah_total"]
tipe = {"kabupaten_kota": "string", **{k: "Int64" for k in kolom[1:]}}
df_file = pd.read_csv(RAW_FILE, encoding="utf-8-sig", skiprows=4, header=None,
                      names=kolom, na_values=["-"], dtype=tipe)
display(df_file)

In [ ]:
# 1d. Identifikasi dataset
print("Jenis data      : TERSTRUKTUR (tabular CSV, baris=wilayah, kolom=atribut tetap)")
print("Jumlah baris    :", df_file.shape[0], "| Jumlah kolom:", df_file.shape[1])
print("Tipe data       :"); print(df_file.dtypes.to_string())
print("Satuan          : unit usaha akomodasi (jumlah hotel non-bintang & akomodasi lainnya)")
print("Cakupan wilayah : 8 kabupaten + 1 kota (Denpasar) + total Provinsi Bali")
print("Cakupan waktu   : tahun 2024 (data tahunan)")
print("Kolom kosong    :", df_file.columns[df_file.isna().all()].tolist(), "(kelompok kamar tidak tersedia di sumber)")

# Validasi internal: jumlah 9 kab/kota harus sama dengan baris total provinsi
kab = df_file[df_file["kabupaten_kota"] != "Provinsi Bali"]
tot = df_file.loc[df_file["kabupaten_kota"] == "Provinsi Bali", "jumlah_total"].iloc[0]
print("Cek total       :", int(kab["jumlah_total"].sum()), "==", int(tot), "->", kab["jumlah_total"].sum() == tot)

# Cek syarat petunjuk: minimal 100 baris
if len(df_file) < 100:
    print(f"\nPERINGATAN: berkas hanya {len(df_file)} baris (< 100 baris sesuai petunjuk 1a).")

In [ ]:
# 1e. Estimasi storage 1 tahun = ukuran berkas x 12 bulan + cadangan 30%
ukuran = RAW_FILE.stat().st_size            # ukuran berkas (byte)
per_tahun = ukuran * 12                     # asumsi: 1 snapshot/bulan, ukuran tetap
total = per_tahun * 1.30                    # tambah cadangan 30% (versi/backup/indeks)
print("Asumsi: 1 unduhan per bulan, ukuran tiap unduhan sama, cadangan 30%")
print(f"Ukuran berkas      : {ukuran:,} byte ({ukuran/1024:.2f} KB)")
print(f"x 12 bulan         : {per_tahun:,} byte")
print(f"+ cadangan 30%     : {total:,.0f} byte = {total/1024:.2f} KB")

## Soal 2 — Lisensi, PII / quasi-identifier, Data Ethics Checklist
Lisensi dan ketentuan portal **harus dibaca langsung dari halaman sumber** lalu dituliskan di sel bawah. Checklist 9 butir diisi di akhir (setelah Soal 3 & 4) karena butuh bukti dari robots.txt dan API.

In [ ]:
# 2a. Catatan lisensi & ketentuan portal (ISI dari halaman sumber, sertakan URL halaman ketentuan)
lisensi_sumber = pd.DataFrame([
    {"sumber": "BPS / Satu Data Bali (berkas)", "lisensi_ketentuan": LISENSI_BPS, "url_ketentuan": "ISI_URL_KETENTUAN_PORTAL", "kewajiban": "sitasi sumber"},
    {"sumber": "Open-Meteo (API)", "lisensi_ketentuan": "CC BY 4.0; API gratis untuk non-komersial", "url_ketentuan": "https://open-meteo.com/en/license", "kewajiban": "atribusi Open-Meteo"},
    {"sumber": "Wikipedia (scraping)", "lisensi_ketentuan": "CC BY-SA 4.0", "url_ketentuan": "https://en.wikipedia.org/wiki/Wikipedia:Text_of_the_Creative_Commons_Attribution-ShareAlike_4.0_International_License", "kewajiban": "atribusi + share-alike"},
])
display(lisensi_sumber)

In [ ]:
# 2b. Identifikasi PII / quasi-identifier pada dataset berkas
# Kata kunci kolom yang lazim PII: nama orang, NIK, alamat, telepon, email, koordinat presisi, dsb.
kata_pii = ["nama", "nik", "alamat", "telp", "telepon", "hp", "email", "lahir", "ktp", "pemilik"]
kolom_pii = [c for c in df_file.columns if any(k in c.lower() for k in kata_pii)]
print("Kolom terindikasi PII langsung :", kolom_pii or "tidak ada")
# Quasi-identifier: atribut yang bila digabung bisa mengidentifikasi (wilayah, waktu)
print("Quasi-identifier               : kabupaten_kota (wilayah) + tahun -> level agregat")
print("Nilai terkecil per sel         :", int(kab["jumlah_total"].min()), "unit (cukup besar, risiko re-identifikasi rendah)")
print("Kesimpulan: data agregat tingkat kab/kota, tanpa data individu/usaha -> tidak ada PII.")

## Soal 3 — Akuisisi via API (Open-Meteo)
BPS tidak dipakai di sini karena butuh API key; sesuai soal, dipakai **API publik yang relevan**: Open-Meteo (cuaca di lokasi dalam dataset, tanpa API key). Titik koordinat = perkiraan pusat tiap kab/kota (**asumsi**, dicatat di metadata).

In [ ]:
# 3a. Koordinat perkiraan pusat tiap kab/kota (asumsi; dokumentasikan di laporan)
LOKASI = {
    "Jembrana": (-8.36, 114.62), "Tabanan": (-8.54, 115.12), "Badung": (-8.65, 115.18),
    "Gianyar": (-8.54, 115.33), "Klungkung": (-8.54, 115.40), "Bangli": (-8.45, 115.35),
    "Karangasem": (-8.45, 115.61), "Buleleng": (-8.11, 115.09), "Denpasar": (-8.65, 115.22),
}
URL_API = "https://archive-api.open-meteo.com/v1/archive"
PERIODE = ("2024-01-01", "2024-01-31")   # periode sama dengan tahun dataset (2024)

# 3b. Ambil data tiap lokasi dengan timeout, retry, dan jeda antar-request
respons_mentah = {}
for nama, (lat, lon) in LOKASI.items():
    print("Mengambil:", nama)
    params = {"latitude": lat, "longitude": lon,
              "start_date": PERIODE[0], "end_date": PERIODE[1],
              "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
              "timezone": "Asia/Makassar"}
    r = get_with_retry(URL_API, params=params)
    respons_mentah[nama] = r.json()
    time.sleep(DELAY)     # jeda sopan antar-request

# 3c. Simpan respons JSON mentah apa adanya ke data/raw, lalu catat provenance
API_RAW = RAW / "openmeteo_archive_bali_2024-01.json"
API_RAW.write_text(json.dumps(respons_mentah, ensure_ascii=False, indent=1), encoding="utf-8")
catat_provenance("Open-Meteo Historical Weather API", "API", URL_API,
                 "CC BY 4.0 (non-komersial)", API_RAW, f"periode {PERIODE}, 9 lokasi, tanpa API key")

In [ ]:
# 3d. Normalisasi JSON bersarang -> tabel datar (satu baris = satu hari per kab/kota)
potongan = []
for nama, js in respons_mentah.items():
    d = pd.DataFrame(js["daily"])                 # kolom: time, temperature_2m_max, ...
    d.insert(0, "kabupaten_kota_api", nama)
    d["latitude"], d["longitude"] = js["latitude"], js["longitude"]
    potongan.append(d)
df_api = pd.concat(potongan, ignore_index=True).rename(columns={"time": "tanggal"})
df_api["tanggal"] = pd.to_datetime(df_api["tanggal"])
print("Baris:", len(df_api), "| Kolom:", df_api.shape[1])
assert len(df_api) >= 30, "Tabel API harus minimal 30 baris"
print("Satuan:", respons_mentah["Badung"]["daily_units"])
display(df_api.head())

## Soal 4 — Web scraping etis (cek robots.txt dulu)
Halaman statis: Wikipedia *List of districts in Bali*. Alur: ambil `robots.txt` -> cek `can_fetch` -> baru ambil halaman -> parse tabel.

In [ ]:
# 4a. Periksa robots.txt (ambil via requests agar tetap memakai timeout & User-Agent kita)
URL_HALAMAN = "https://en.wikipedia.org/wiki/List_of_districts_in_Bali"
URL_ROBOTS = "https://en.wikipedia.org/robots.txt"
r_robots = get_with_retry(URL_ROBOTS)
ROBOTS_RAW = RAW / "wikipedia_robots.txt"
ROBOTS_RAW.write_text(r_robots.text, encoding="utf-8")       # simpan sebagai bukti kepatuhan

rp = robotparser.RobotFileParser()
rp.parse(r_robots.text.splitlines())
boleh = rp.can_fetch(USER_AGENT, URL_HALAMAN)
print("User-Agent        :", USER_AGENT)
print("URL halaman       :", URL_HALAMAN)
print("Diizinkan robots? :", boleh)
print("Crawl-delay       :", rp.crawl_delay(USER_AGENT))
catat_provenance("Wikipedia robots.txt", "bukti kepatuhan", URL_ROBOTS, "n/a", ROBOTS_RAW,
                 f"can_fetch={boleh}")
assert boleh, "robots.txt melarang halaman ini -> JANGAN scraping, pilih halaman lain"
time.sleep(DELAY)

In [ ]:
# 4b. Ambil halaman, simpan HTML mentah, lalu catat provenance
r_html = get_with_retry(URL_HALAMAN)
HTML_RAW = RAW / "wikipedia_list_districts_bali.html"
HTML_RAW.write_text(r_html.text, encoding="utf-8")
catat_provenance("Wikipedia - List of districts in Bali", "web scraping", URL_HALAMAN,
                 "CC BY-SA 4.0", HTML_RAW, "halaman statis, 1 request")

# 4c. Ambil semua tabel dengan pd.read_html, lalu pilih tabel yang memuat kata "Badung"
tabel_semua = pd.read_html(io.StringIO(r_html.text))
print("Jumlah tabel di halaman:", len(tabel_semua))
for i, t in enumerate(tabel_semua):
    print(i, t.shape, list(map(str, t.columns))[:6])

In [ ]:
# 4d. Pilih tabel (otomatis: tabel terbesar yang memuat "Badung"). Jika salah, set IDX_TABEL manual.
kandidat = [i for i, t in enumerate(tabel_semua) if t.astype(str).apply(lambda c: c.str.contains("Badung")).any().any()]
IDX_TABEL = max(kandidat, key=lambda i: len(tabel_semua[i]))
df_web = tabel_semua[IDX_TABEL].copy()
# Ratakan header bertingkat (MultiIndex) jika ada
df_web.columns = [" ".join(map(str, c)).strip() if isinstance(c, tuple) else str(c) for c in df_web.columns]
print("Tabel terpilih:", IDX_TABEL, "| baris:", len(df_web))
display(df_web.head(10))
# CEK VISUAL: pastikan tabel berisi data kecamatan/kab-kota dengan kolom angka (luas/penduduk)

In [ ]:
# 4e. Bukti pemeriksaan kepatuhan (ringkasan untuk laporan)
bukti_scraping = {
    "robots_txt_url": URL_ROBOTS, "robots_disimpan": str(ROBOTS_RAW),
    "can_fetch": boleh, "user_agent": USER_AGENT, "timeout_detik": TIMEOUT,
    "jeda_detik": DELAY, "jumlah_request_ke_halaman": 1, "login_dibutuhkan": False,
    "halaman_statis": True, "tanggal": datetime.now().astimezone().isoformat(timespec="seconds"),
}
print(json.dumps(bukti_scraping, indent=2))

## Soal 2 (lanjutan) — Data Ethics Checklist 9 butir
> Jika dosen punya daftar 9 butir resmi di materi kuliah, sesuaikan kolom `butir` dengan daftar itu; bukti di bawah tetap bisa dipakai.

In [ ]:
# Checklist 9 butir + bukti (sebagian terisi otomatis dari hasil sel sebelumnya)
checklist = pd.DataFrame([
    (1, "Sumber sah & diizinkan (bukan platform terlarang)", "Ya", "BPS/portal resmi, Open-Meteo, Wikipedia; tidak memakai Booking/Agoda/Traveloka/tiket.com/Airbnb/Tripadvisor/Google Maps/medsos"),
    (2, "Lisensi dataset diperiksa & dicatat", "Ya", "Lihat tabel lisensi_sumber dan provenance.csv"),
    (3, "Ketentuan penggunaan portal dipatuhi", "Ya", "URL ketentuan tercatat di tabel lisensi_sumber"),
    (4, "Tidak ada PII / sudah dinilai", "Ya", f"Kolom PII: {kolom_pii or 'tidak ada'}; data agregat kab/kota"),
    (5, "Minimisasi data (hanya yang perlu)", "Ya", "Hanya 3 variabel cuaca + data wilayah yang relevan"),
    (6, "robots.txt diperiksa sebelum scraping", "Ya", f"can_fetch={boleh}; berkas {ROBOTS_RAW}"),
    (7, "Rate limiting, timeout, User-Agent identitas", "Ya", f"timeout={TIMEOUT}s, jeda={DELAY}s, UA berisi email kampus"),
    (8, "Kredensial tidak tertulis di notebook", "Ya", "Email/key dibaca dari Colab Secrets atau .env; Open-Meteo tanpa key"),
    (9, "Provenance & atribusi sumber lengkap", "Ya", "provenance.csv (URL, tanggal akses, lisensi, MD5)"),
], columns=["no", "butir", "status", "bukti"])
checklist.to_csv(PROV / "data_ethics_checklist.csv", index=False)
display(checklist)

## Soal 5 — Penggabungan (merge) minimal 2 sumber + validasi
Sumber digabung: **berkas BPS + API Open-Meteo + hasil scraping** (3 sumber). Kunci gabung = nama wilayah ternormalisasi. Memakai `validate="one_to_one"` dan `indicator=True` sebagai bukti tidak ada duplikasi.

In [ ]:
# 5a. Kunci gabung: samakan penulisan nama wilayah ("Kab. Badung", "Badung Regency", "Kota Denpasar" -> "badung", "denpasar")
def buat_kunci(s):
    s = re.sub(r"\[.*?\]", "", str(s)).lower()                     # buang catatan kaki [1]
    s = re.sub(r"\b(kabupaten|kab\.?|kota|regency|city|provinsi)\b", "", s)
    return re.sub(r"[^a-z]", "", s)

# Sumber 1: berkas (hanya 9 kab/kota, tanpa baris total provinsi)
a = df_file[df_file["kabupaten_kota"] != "Provinsi Bali"].copy()
a["kunci"] = a["kabupaten_kota"].map(buat_kunci)

# Sumber 2: API -> ringkas per kab/kota agar 1 baris per wilayah
b = (df_api.groupby("kabupaten_kota_api", as_index=False)
     .agg(suhu_maks_rata2=("temperature_2m_max", "mean"),
          suhu_min_rata2=("temperature_2m_min", "mean"),
          hujan_total_mm=("precipitation_sum", "sum"),
          hari_data=("tanggal", "count")))
b["kunci"] = b["kabupaten_kota_api"].map(buat_kunci)

# Sumber 3: scraping -> cari kolom wilayah (memuat "Badung") & ubah kolom angka, lalu ringkas per wilayah
kol_wilayah = next(c for c in df_web.columns if df_web[c].astype(str).str.contains("Badung").any())
w = df_web.copy()
w["kunci"] = w[kol_wilayah].map(buat_kunci)
for c in w.columns.difference([kol_wilayah, "kunci"]):
    w[c] = pd.to_numeric(w[c].astype(str).str.replace(r"\[.*?\]|[,\s]", "", regex=True), errors="coerce")
angka = [c for c in w.columns if c not in (kol_wilayah, "kunci") and w[c].notna().mean() > 0.5]
c_ = (w[w["kunci"].isin(a["kunci"])].groupby("kunci", as_index=False)[angka].sum(min_count=1))
c_["jumlah_baris_web"] = w[w["kunci"].isin(a["kunci"])].groupby("kunci").size().values
print("Kolom angka dari scraping:", angka)
print("Kunci cocok  file/api/web:", set(a.kunci) == set(b.kunci), set(a.kunci) <= set(c_.kunci))

In [ ]:
# 5b. Merge dengan validate (error otomatis jika ada duplikasi kunci) + indicator
m1 = a.merge(b, on="kunci", how="left", validate="one_to_one", indicator="_cek_api")
m2 = m1.merge(c_, on="kunci", how="left", validate="one_to_one", indicator="_cek_web")
print(m2[["_cek_api", "_cek_web"]].apply(pd.Series.value_counts))   # semua harus 'both'
hasil = m2.drop(columns=["_cek_api", "_cek_web", "kabupaten_kota_api"])
display(hasil)

In [ ]:
# 5c. BUKTI tidak ada duplikasi baris
assert hasil["kunci"].is_unique,            "Kunci duplikat!"
assert hasil.duplicated().sum() == 0,        "Ada baris duplikat!"
assert len(hasil) == len(a),                 "Jumlah baris berubah setelah merge!"
print("Baris sebelum merge :", len(a))
print("Baris sesudah merge :", len(hasil))
print("Duplikat kunci      :", hasil['kunci'].duplicated().sum())
print("Duplikat baris      :", hasil.duplicated().sum())
print("Nilai kosong per kolom:"); print(hasil.isna().sum().to_string())

In [ ]:
# 5d. Simpan hasil gabungan + metadata + provenance + checklist
HASIL_CSV = PROC / "akomodasi_cuaca_wilayah_bali_merged.csv"
hasil.to_csv(HASIL_CSV, index=False)

metadata = {
    "judul": "Gabungan akomodasi non-bintang, cuaca, dan data wilayah Bali",
    "dibuat": datetime.now().astimezone().isoformat(timespec="seconds"),
    "sumber_digabung": ["berkas BPS (CSV)", "Open-Meteo API", "Wikipedia (scraping)"],
    "kunci_gabung": "nama wilayah ternormalisasi (kolom 'kunci')",
    "parameter_merge": {"how": "left", "validate": "one_to_one", "indicator": True},
    "jumlah_baris": int(len(hasil)), "jumlah_kolom": int(hasil.shape[1]),
    "kolom": list(hasil.columns), "md5_hasil": md5_file(HASIL_CSV),
    "asumsi": ["koordinat = perkiraan pusat wilayah", f"periode cuaca {PERIODE}"],
    "bukti_tanpa_duplikasi": {"duplikat_kunci": 0, "duplikat_baris": 0},
}
(PROC / "metadata_merged.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
pd.DataFrame(provenance).to_csv(PROV / "provenance.csv", index=False)
display(pd.DataFrame(provenance)[["sumber", "metode", "tanggal_akses", "lisensi", "md5", "ukuran_byte"]])
print(json.dumps(metadata, indent=2, ensure_ascii=False))

In [ ]:
# 5e. Cek akhir: daftar berkas yang harus dikumpulkan
for p in sorted(Path("data").rglob("*")):
    if p.is_file():
        print(p, f"({p.stat().st_size:,} byte)")

## Checklist pengumpulan
- [ ] Semua sel sudah dijalankan (Runtime -> Run all), lalu **File -> Save a copy in GitHub** atau bagikan link Colab (akses "Anyone with the link")
- [ ] Folder `data/raw` (CSV, JSON API, robots.txt, HTML) tidak diubah
- [ ] `data/processed` (hasil merge + metadata), `data/provenance` (provenance + checklist)
- [ ] Kirim URL ke ELSA, siapkan demo alur akuisisi